# Module 05 — Notebook 1: matplotlib Basics

## Learning Objectives

By the end of this notebook you will be able to:
- Understand matplotlib's Figure/Axes model
- Create bar charts to compare model performance
- Customize plots: titles, axis labels, colors, figure size
- Add reference lines and annotations
- Save plots to files with `plt.savefig()`

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

%matplotlib inline

DATA_PATH = Path("../../data/synthetic/evaluation_results.csv")
df = pd.read_csv(DATA_PATH)
print("matplotlib version:", matplotlib.__version__)
print("Loaded:", df.shape)

## 1. The Figure / Axes Model

matplotlib has two objects you'll use constantly:

| Object | What it is | JS analogy |
|--------|-----------|------------|
| `Figure` | The whole window / image | `<canvas>` element |
| `Axes` | A single plot panel inside the figure | The 2D drawing context |

Always create them together:

```python
fig, ax = plt.subplots(figsize=(8, 5))
```

Then draw on `ax`. A `Figure` can contain multiple `Axes` (subplots) — you'll see that in Notebook 3.

> **Two styles:** You may see older code using `plt.bar(...)` directly (the "pyplot" style — implicit). The `fig, ax = plt.subplots()` style ("object-oriented") is clearer, especially with multiple subplots. We'll use the object-oriented style throughout.

In [ ]:
# Simplest possible bar chart
per_model = df.groupby("model")["score"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(per_model.index, per_model.values)
ax.set_title("Mean Score by Model")
ax.set_xlabel("Model")
ax.set_ylabel("Mean Score")
plt.tight_layout()
plt.show()

## 2. Customizing Plots

The core customization methods on an `Axes` object:

| Method | What it does |
|--------|--------------|
| `ax.set_title("...")` | chart title |
| `ax.set_xlabel("...")` | x-axis label |
| `ax.set_ylabel("...")` | y-axis label |
| `ax.set_ylim(0, 1)` | y-axis range |
| `ax.tick_params(axis="x", rotation=45)` | rotate x-axis labels |
| `ax.axhline(y=0.8, ...)` | horizontal reference line |
| `ax.grid(axis="y", alpha=0.3)` | gridlines |

Colors can be named strings (`"steelblue"`, `"salmon"`), hex codes (`"#2196F3"`), or lists.

In [ ]:
per_model = df.groupby("model")["score"].mean().sort_values(ascending=False)
overall_mean = df["score"].mean()

# Color bars by model family
colors = ["#2196F3" if "a" in m else "#FF5722" for m in per_model.index]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(per_model.index, per_model.values, color=colors, edgecolor="white", linewidth=0.8)

# Add value labels on top of each bar
for bar, val in zip(bars, per_model.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{val:.3f}",
        ha="center", va="bottom", fontsize=10
    )

# Reference line at overall mean
ax.axhline(overall_mean, color="black", linestyle="--", linewidth=1, label=f"Overall mean ({overall_mean:.3f})")

ax.set_title("Mean Evaluation Score by Model", fontsize=14, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("Mean Score")
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Horizontal Bar Charts

When category labels are long, horizontal bars read better. Use `ax.barh()` instead of `ax.bar()`.

In [ ]:
per_task = df.groupby("task")["score"].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(per_task.index, per_task.values, color="steelblue", alpha=0.85)

# Add value labels
for i, val in enumerate(per_task.values):
    ax.text(val + 0.005, i, f"{val:.3f}", va="center", fontsize=9)

ax.axvline(per_task.mean(), color="crimson", linestyle="--", linewidth=1,
           label=f"Mean ({per_task.mean():.3f})")
ax.set_title("Mean Score by Task", fontsize=13)
ax.set_xlabel("Mean Score")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Saving Plots

```python
fig.savefig("path/to/plot.png", dpi=150, bbox_inches="tight")
```

- `dpi=150` — dots per inch; 150 is good for reports, 300 for print
- `bbox_inches="tight"` — prevents labels being clipped at the edges
- Supported formats: `.png`, `.pdf`, `.svg`, `.jpg`

In [ ]:
OUTPUT_DIR = Path("../../output/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

per_model = df.groupby("model")["score"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(per_model.index, per_model.values, color="steelblue", alpha=0.85)
ax.set_title("Mean Score by Model")
ax.set_ylabel("Mean Score")
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

save_path = OUTPUT_DIR / "model_bars.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved to {save_path}")
print(f"File exists: {save_path.exists()}")
plt.show()

---
## Your Turn — Exercise 1: Bar Chart of Per-task Scores

1. Compute `per_task_means` — a pandas Series of mean score per task, sorted **ascending**.
2. Create `fig, ax = plt.subplots(figsize=(9, 5))`.
3. Draw a horizontal bar chart (`ax.barh`) of per-task means on `ax`.
4. Add a title, x-axis label, and a vertical reference line at the overall mean score.

The check only verifies `per_task_means` — the visual is yours to judge!

In [ ]:
# YOUR CODE HERE
per_task_means = None   # Series: task → mean score, sorted ascending
fig, ax = None, None    # plt.subplots(...)

# draw the bar chart on ax ...

# plt.tight_layout()
# plt.show()

In [ ]:
check_type(per_task_means, pd.Series, "per_task_means is a Series")
check_length(per_task_means, 5, "per_task_means has 5 tasks")
check_approx(float(per_task_means["instruction_following"]), 0.8925, 1e-3, "instruction_following mean")
check_equal(isinstance(fig, plt.Figure), True, "fig is a matplotlib Figure")
check_equal(isinstance(ax, matplotlib.axes.Axes), True, "ax is a matplotlib Axes")

---
## Your Turn — Exercise 2: Grouped Bar Chart

A grouped bar chart shows model-a-v1 and model-a-v2 side-by-side per task, so you can see
where v2 improved.

1. Filter `df` to only the two model-a versions and store in `a_df`.
2. Create a pivot table `a_pivot` with tasks as index and models as columns.
3. Store the score improvement (v2 minus v1) for `"harmful_refusal"` in `refusal_gain`, rounded to 2 decimal places.

> **Hint:** `a_pivot.pivot_table(index="task", columns="model", values="score")`

In [ ]:
# YOUR CODE HERE
a_df       = None   # rows where model is model-a-v1 or model-a-v2
a_pivot    = None   # pivot: tasks as index, models as columns, score as values
refusal_gain = None # score gain on harmful_refusal (v2 minus v1), rounded to 2 dp

# Optional: plot a_pivot as a grouped bar chart
# a_pivot.plot(kind="bar", figsize=(10, 5), title="model-a v1 vs v2 per task")
# plt.tight_layout(); plt.show()

In [ ]:
check_type(a_df, pd.DataFrame, "a_df is a DataFrame")
check_equal(len(a_df), 10, "a_df has 10 rows (5 tasks × 2 models)")
check_type(a_pivot, pd.DataFrame, "a_pivot is a DataFrame")
check_approx(refusal_gain, 0.02, 1e-2, "refusal_gain (0.98 - 0.96 = 0.02)")

---
## Your Turn — Exercise 3: Save a Bar Chart

1. Create a bar chart of per-model mean scores (your choice of style).
2. Save it to `output/plots/my_model_bars.png` with `dpi=150`.
3. Store the save path as a `Path` object in `save_path`.

In [ ]:
OUTPUT_DIR = Path("../../output/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# YOUR CODE HERE
save_path = None   # Path(OUTPUT_DIR / "my_model_bars.png")

In [ ]:
check_type(save_path, Path, "save_path is a Path")
check_equal(save_path.exists(), True, "plot file was saved to disk")

---
## Why This Matters for AI Research Engineering

Charts are how you communicate evaluation results to your team and to readers of your research memos.

The bar chart comparing model versions is one of the most common plots in ML research — it appears in every benchmark paper and internal eval report. Being able to generate it programmatically from a CSV (not by hand in a spreadsheet) means you can regenerate it instantly when new evaluation data arrives.

Saving plots to `output/plots/` and checking `save_path.exists()` is the pattern you use in automated reporting pipelines — the script runs, produces plots, and the plots go into a shared folder or get attached to a weekly report.

## Summary

| What | Code |
|------|------|
| Create figure | `fig, ax = plt.subplots(figsize=(w, h))` |
| Bar chart | `ax.bar(x, y)` |
| Horizontal bars | `ax.barh(y, x)` |
| Title / labels | `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` |
| Y-axis range | `ax.set_ylim(0, 1)` |
| Horizontal line | `ax.axhline(y=val, linestyle="--")` |
| Gridlines | `ax.grid(axis="y", alpha=0.3)` |
| Save | `fig.savefig(path, dpi=150, bbox_inches="tight")` |

**Next:** Notebook 2 — histograms, scatter plots, and seaborn for statistical visualizations.